In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql import functions as F

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/06 05:29:44 WARN Utils: Your hostname, codespaces-029a01 resolves to a loopback address: 127.0.0.1; using 10.0.0.158 instead (on interface eth0)
25/03/06 05:29:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/06 05:29:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df_green = spark.read.parquet('data/pq/green/*/*')

In [22]:
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

In [5]:
df_green = df_green \
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")

In [23]:
df_yellow = df_yellow \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

In [9]:
df_green.createOrReplaceTempView('green_data')

In [24]:
df_yellow.createOrReplaceTempView('yellow_data')

In [25]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', pickup_datetime) AS revenue_hour, 
    PULocationID AS revenue_zone,
    -- Revenue calculation 
    SUM(total_amount) AS revenue_total_amount,
    COUNT(1) as number_records
FROM
    green_data
WHERE
    YEAR(pickup_datetime) >= 2020 
GROUP BY
    1, 2
""")

In [26]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', pickup_datetime) AS revenue_hour, 
    PULocationID AS revenue_zone,
    -- Revenue calculation 
    SUM(total_amount) AS revenue_total_amount,
    COUNT(1) as number_records
FROM
    yellow_data
WHERE
    YEAR(pickup_datetime) >= 2020 
GROUP BY
    1, 2
""")

In [27]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

In [28]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [34]:
df_green_tmp = df_green_revenue \
    .withColumnRenamed('revenue_total_amount', 'green_total_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_tmp = df_yellow_revenue \
    .withColumnRenamed('revenue_total_amount', 'yellow_total_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [35]:
df_revenue = df_green_tmp.join(df_yellow_tmp, on=['revenue_hour','revenue_zone'], how='outer')

In [ ]:
df_revenue.show()

In [37]:
df_zones = spark.read \
    .option("header", "true") \
    .csv('../../Week4-dbt/seeds/taxi_zones_lookup.csv')

df_zones.show()

+----------+-------------+--------------------+------------+
|locationid|      borough|                zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [38]:
df_join = df_revenue.join(df_zones, df_zones.locationid == df_revenue.revenue_zone)
df_join.drop('locationid').show()

[Stage 46:>                                                         (0 + 1) / 1]

+-------------------+------------+------------------+--------------------+-------------------+---------------------+---------+--------------------+------------+
|       revenue_hour|revenue_zone|green_total_amount|green_number_records|yellow_total_amount|yellow_number_records|  borough|                zone|service_zone|
+-------------------+------------+------------------+--------------------+-------------------+---------------------+---------+--------------------+------------+
|2020-01-01 00:00:00|           3|              NULL|                NULL|               25.0|                    1|    Bronx|Allerton/Pelham G...|   Boro Zone|
|2020-01-01 00:00:00|           4|              NULL|                NULL| 1004.3000000000002|                   57|Manhattan|       Alphabet City| Yellow Zone|
|2020-01-01 00:00:00|           7| 769.7299999999996|                  45|  455.1700000000001|                   38|   Queens|             Astoria|   Boro Zone|
|2020-01-01 00:00:00|          12|